# 03 - Pixel table for each field-year

For every 5 m pixel of each kriged yield raster, the soil properties, topography, nitrogen zone,
growing-season ETa and soil moisture (mean of the October-April monthly layers) and P are read at the
pixel centre. One csv is written per field-year into `data/model_input/ASP` or `.../BAU`, which is the
input of notebooks 04-06.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.colors import Normalize

sys.path.append("../src")
from yieldml import ASP_FIELDS, Y_MULT

In [ ]:
RAW = Path("../data/raw")
YIELD_DIR = RAW / "yield"                      # <YYYY>/<Field>_<YYYY>_Wheat.tif, bu/ac, 5 m
SOIL_DIR = RAW / "soil_rasters"                # 'Clay (0-15 cm).tif', ...
TOPO_DIR = RAW / "topography"
N_DIR = RAW / "nitrogen"                       # Nitrogen_<YYYY>.tif
MONTHLY = Path("../data/processed/monthly/aligned")
P_CSV = Path("../data/processed/precipitation_oct_apr15.csv")
OUT = Path("../data/model_input")

SOIL = [f"{p} ({d} cm)" for p in ["Carbon"] for d in ["0-15", "15-30"]] + \
       [f"{p} ({d} cm)" for p in ["Clay", "Sand", "Silt"] for d in ["0-15", "15-30", "30-60", "60-90", "90-120"]] + \
       [f"{p} ({d} cm)" for p in ["OM", "pH"] for d in ["0-15", "15-30"]]
TOPO = {"Elevation": "DEM.tif", "Aspect": "Aspect.tif", "Curvature": "Curvature.tif",
        "Slope": "Slope.tif", "TPI": "TPI.tif"}
MONTHLY_VARS = {"ETa": "ETa", **{f"SM {d} cm": f"SM{d}" for d in (30, 60, 90, 120, 150, 180)}}
MONTHS = ["10", "11", "12", "01", "02", "03", "04"]    # October to April

precip = pd.read_csv(P_CSV).set_index("Year")["P_Oct_to_Apr15_mm"]

## Sampling

Every feature raster is read at the centre of each valid yield pixel (nearest pixel). Points that fall
outside a raster get NaN.

In [ ]:
def sample(path, xs, ys):
    with rasterio.open(path) as src:
        band = src.read(1).astype("float64")
        rows, cols = zip(*[src.index(x, y) for x, y in zip(xs, ys)])
        rows, cols = np.array(rows), np.array(cols)
        inside = (rows >= 0) & (rows < src.height) & (cols >= 0) & (cols < src.width)
        out = np.full(len(xs), np.nan)
        out[inside] = band[rows[inside], cols[inside]]
    return out


def field_year_table(yield_tif, year):
    with rasterio.open(yield_tif) as src:
        yld = src.read(1).astype("float64")
        valid = ~np.isnan(yld) if src.nodata is None else (yld != src.nodata) & ~np.isnan(yld)
        r, c = np.where(valid)
        xs, ys = src.transform * (c + 0.5, r + 0.5)

    t = pd.DataFrame({"latitude": ys, "longitude": xs, "yield": yld[r, c]})
    for name in SOIL:
        t[name] = sample(SOIL_DIR / f"{name}.tif", xs, ys)
    for name, fname in TOPO.items():
        t[name] = sample(TOPO_DIR / fname, xs, ys)
    t["Nitrogen"] = sample(N_DIR / f"Nitrogen_{year}.tif", xs, ys)

    # growing-season mean of the monthly layers, October (year - 1) to April (year)
    for name, var in MONTHLY_VARS.items():
        layers = []
        for m in MONTHS:
            y = year - 1 if m in ("10", "11", "12") else year
            path = MONTHLY / var / f"{var}_{y}-{m}.tif"
            if path.exists():             # months without any image were not written
                layers.append(sample(path, xs, ys))
        t[name] = np.nanmean(np.vstack(layers), axis=0)

    t["Precipitation"] = precip.loc[year]
    return t

In [ ]:
for tif in sorted(YIELD_DIR.glob("*/*.tif")):
    field, year = tif.stem.split("_")[0], int(tif.stem.split("_")[1])
    group = "ASP" if field in ASP_FIELDS else "BAU"
    table = field_year_table(tif, year)
    (OUT / group).mkdir(parents=True, exist_ok=True)
    table.to_csv(OUT / group / f"{tif.stem}.csv", index=False)
    print(f"{tif.stem}: {len(table)} pixels -> {group}")

## Yield maps (Supplementary Fig. S1)

In [ ]:
def to_grid(df, pix=5.0):
    xs, ys = np.unique(df["longitude"]), np.unique(df["latitude"])
    xi, yi = {v: i for i, v in enumerate(xs)}, {v: i for i, v in enumerate(ys)}
    arr = np.full((len(ys), len(xs)), np.nan)
    for x, y, v in zip(df["longitude"], df["latitude"], df["yield"]):
        arr[yi[y], xi[x]] = v
    return arr, [xs.min() - pix / 2, xs.max() + pix / 2, ys.min() - pix / 2, ys.max() + pix / 2]


files = sorted(OUT.glob("*/*.csv"))
for year in sorted({f.stem.split("_")[1] for f in files}):
    year_files = sorted((f for f in files if f.stem.split("_")[1] == year), key=lambda f: f.stem)
    tables = {f.stem.split("_")[0]: pd.read_csv(f, usecols=["latitude", "longitude", "yield"]).dropna()
              for f in year_files}
    for t in tables.values():
        t["yield"] *= Y_MULT
    norm = Normalize(min(t["yield"].min() for t in tables.values()),
                     max(t["yield"].max() for t in tables.values()))

    fig, axes = plt.subplots(3, 2, figsize=(12, 10))
    im = None
    for ax, (name, t) in zip(axes.ravel(), tables.items()):
        arr, extent = to_grid(t)
        im = ax.imshow(arr, extent=extent, origin="lower", interpolation="nearest", norm=norm)
        ax.set_title(name, fontsize=12, pad=2)
    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect("equal")
        for s in ax.spines.values():
            s.set_visible(False)
    fig.colorbar(im, cax=fig.add_axes([0.92, 0.12, 0.015, 0.76]), label="Yield (kg/ha)")
    fig.suptitle(f"Yield maps - {year}", fontsize=16, y=0.98)
    fig.subplots_adjust(left=0.03, right=0.9, top=0.95, bottom=0.05, wspace=0.05, hspace=0.05)
    plt.savefig(f"../results/figures/supp_fig_s1_yield_{year}.png", dpi=300)
    plt.show()